In [ ]:
# 1. Install Document Extraction & Parsing (Lightning-fast Markdown approach)
!pip install pymupdf4llm langchain-text-splitters pypdf==4.2.0

# 2. Install Vector Database
!pip install chromadb==0.5.0

# 3. Install Embeddings & Deep Learning Foundations
!pip install torch>=2.0.0 sentence-transformers==3.0.1 hf_xet

# 4. Install AI Orchestration Frameworks & Gateways
!pip install langchain==0.2.5 langchain-community==0.2.5 openai==1.35.3 litellm==1.41.1

# 5. Install Evaluation, Observability & SRE Metrics
!pip install deepeval==4.1.1 langsmith

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import json
import hashlib
import re
import pymupdf4llm
import chromadb
from chromadb.utils import embedding_functions
from litellm import completion
from langchain_text_splitters import RecursiveCharacterTextSplitter
from google.colab import userdata, drive

# CONFIRMED-WORKING config from Phase 1 — do not swap silently
MODEL_NAME = "gemini/gemini-3.5-flash"
EMBED_MODEL = "all-MiniLM-L6-v2"


# ---------------------------------------------------------
# 0. DRIVE MOUNT — HARD VERIFICATION
# ---------------------------------------------------------
# os.makedirs() will happily create a LOCAL folder tree with the same path
# if /content/drive isn't actually mounted — it doesn't error, it just
# silently writes somewhere that vanishes on the next runtime restart.
# That is almost certainly what created your "duplicate" folder: a real
# Drive mount, then a later cell run where the mount had dropped, quietly
# writing a second, local-only copy at the identical-looking path.

def ensure_drive_mounted(mount_point="/content/drive"):
    if not os.path.ismount(mount_point):
        print("🔗 Mounting Google Drive...")
        drive.mount(mount_point)
    if not os.path.ismount(mount_point):
        raise RuntimeError(
            f"❌ Google Drive did NOT mount at {mount_point}. Refusing to "
            f"proceed — continuing here would silently create a local, "
            f"non-persistent shadow folder instead of writing to your real Drive."
        )
    print(f"✅ Google Drive verified as mounted at {mount_point}")

def verify_base_path(base_path):
    """Sanity-check that base_path actually contains your real data, not an
    empty shadow directory. Fails loud instead of silently proceeding."""
    regulatory_dir = os.path.join(base_path, "regulatory")
    if not os.path.isdir(regulatory_dir):
        raise RuntimeError(
            f"❌ {regulatory_dir} does not exist. This base_path looks wrong "
            f"or points to a freshly-created empty folder — check the exact "
            f"path in your Drive UI and compare it to base_path below."
        )
    pdfs = [f for f in os.listdir(regulatory_dir) if f.endswith(".pdf")]
    if not pdfs:
        raise RuntimeError(
            f"❌ {regulatory_dir} exists but has no PDFs in it. This is "
            f"probably the wrong folder, not your real data."
        )
    print(f"✅ base_path verified — found {len(pdfs)} PDF(s) in {regulatory_dir}")

# ---------------------------------------------------------
# 1. CACHING & INGESTION
# ---------------------------------------------------------

def hash_file(filepath):
    hasher = hashlib.md5()
    with open(filepath, 'rb') as f:
        hasher.update(f.read())
    return hasher.hexdigest()

def ingest_regulatory_pdf(pdf_path, chunk_offset=0):
    print(f"⏳ Extracting {os.path.basename(pdf_path)} to Markdown...")
    md_text = pymupdf4llm.to_markdown(pdf_path)
    section_pattern = re.compile(r'(\n\*\*\d+[A-Z]?\.\s+[^\*]+\*\*)')
    parts = section_pattern.split(md_text)
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=150)
    chunks = []
    chunk_idx = chunk_offset

    if parts[0].strip():
        for split_txt in text_splitter.split_text(parts[0].strip()):
            chunks.append({
                "text": split_txt,
                "metadata": {"source": os.path.basename(pdf_path), "doc_type": "regulatory",
                             "section_title": "Preamble / TOC", "chunk_id": f"chunk_{chunk_idx}"}
            })
            chunk_idx += 1

    for i in range(1, len(parts), 2):
        header_raw, body_raw = parts[i].strip(), parts[i + 1].strip()
        clean_title = header_raw.replace('**', '').split('.—')[0].split('—')[0].strip()
        for split_txt in text_splitter.split_text(f"{header_raw}\n{body_raw}"):
            chunks.append({
                "text": split_txt,
                "metadata": {"source": os.path.basename(pdf_path), "doc_type": "regulatory",
                             "section_title": clean_title, "chunk_id": f"chunk_{chunk_idx}"}
            })
            chunk_idx += 1
    return chunks, chunk_idx

def ingest_synthetic_json(json_path):
    print(f"⏳ Extracting synthetic records from {os.path.basename(json_path)}...")
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    if not isinstance(data, dict):
        return []

    chunks = []
    for category in ['maintenance_logs', 'incident_reports', 'work_permits']:
        for item in data.get(category, []):
            # FIX: use the real record ID field, which differs per category —
            # item.get('id', ...) never matched anything before
            record_id = (
                item.get('log_id') or item.get('incident_id') or item.get('permit_id')
                or hashlib.md5(str(item).encode()).hexdigest()[:8]
            )
            text_lines = [f"Record Type: {category}"] + [f"{k}: {v}" for k, v in item.items()]
            chunk_id = f"{category}_{record_id}"
            chunks.append({
                "text": "\n".join(text_lines),
                "metadata": {
                    "source": os.path.basename(json_path),
                    "doc_type": category,
                    "equipment_id": item.get("equipment_id", "UNKNOWN"),
                    "record_id": record_id,          # NEW — needed for Phase 3 graph linkage
                    "chunk_id": chunk_id,
                }
            })
    return chunks

def build_vector_store(data_dirs, cache_path, chroma_path):
    manifest_path = cache_path.replace('.json', '_manifest.json')

    if os.path.exists(manifest_path) and os.path.exists(cache_path):
        print("📁 Loading existing text database from Drive cache...")
        with open(manifest_path, 'r', encoding='utf-8') as f:
            manifest = json.load(f)
        with open(cache_path, 'r', encoding='utf-8') as f:
            all_chunks = json.load(f)
    else:
        print("✨ No cache found. Starting fresh.")
        manifest, all_chunks = {}, []

    new_chunks = []
    chunk_counter = len(all_chunks)
    requires_db_update = False

    for d in data_dirs:
        if not os.path.exists(d):
            continue
        for f in sorted(os.listdir(d)):  # sorted = deterministic processing order
            filepath = os.path.join(d, f)
            if (f.endswith('.pdf') or (f.endswith('.json') and 'synthetic' in d)) and os.path.getsize(filepath) > 0:
                current_hash = hash_file(filepath)
                if manifest.get(f) != current_hash:
                    print(f"🔄 Processing new/modified file: {f}")
                    requires_db_update, manifest[f] = True, current_hash
                    if f.endswith('.pdf'):
                        c, chunk_counter = ingest_regulatory_pdf(filepath, chunk_counter)
                        new_chunks.extend(c)
                    else:
                        new_chunks.extend(ingest_synthetic_json(filepath))
                else:
                    print(f"✅ Skipping unchanged file: {f}")

    local_emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)
    chroma_client = chromadb.PersistentClient(path=chroma_path)
    collection = chroma_client.get_or_create_collection(name="industrial_brain_v2", embedding_function=local_emb_fn)

    if requires_db_update:
        print("💾 Writing new chunks + updated cache to Drive...")
        all_chunks.extend(new_chunks)
        with open(cache_path, 'w', encoding='utf-8') as f:
            json.dump(all_chunks, f, ensure_ascii=False, indent=4)
        with open(manifest_path, 'w', encoding='utf-8') as f:
            json.dump(manifest, f, ensure_ascii=False, indent=4)
        collection.upsert(  # upsert, not add — tolerates re-runs without crashing on dupe IDs
            documents=[d["text"] for d in new_chunks],
            metadatas=[d["metadata"] for d in new_chunks],
            ids=[d["metadata"]["chunk_id"] for d in new_chunks]
        )
        print(f"🚀 {len(new_chunks)} new chunks added.")
    else:
        print("🚀 No changes detected. Skipping ingestion entirely.")
        if collection.count() == 0 and len(all_chunks) > 0:
            print("💾 Re-hydrating vector store from cache (persisted DB was empty)...")
            collection.upsert(
                documents=[d["text"] for d in all_chunks],
                metadatas=[d["metadata"] for d in all_chunks],
                ids=[d["metadata"]["chunk_id"] for d in all_chunks]
            )

    print(f"📊 Total chunks in collection: {collection.count()}")
    return collection

# ---------------------------------------------------------
# 2. RETRIEVAL / GENERATION / VERIFICATION TOOLS
# ---------------------------------------------------------

def expand_query(query, doc_type=None):
    # FIX: don't force legal-term expansion on non-regulatory queries
    if doc_type == "regulatory":
        instruction = ("Expand this query into related legal/regulatory terms, section "
                        "references, and synonyms likely found in Indian industrial safety law.")
    elif doc_type is None:
        instruction = ("Expand this query into terms covering BOTH regulatory/legal language "
                        "AND operational terms (permit types, equipment tags, record fields) — "
                        "this query may need to match documents of either kind.")
    else:
        instruction = ("Expand this query into related equipment, maintenance, and operational "
                        "terms (equipment tags, symptoms, actions). Do not add legal jargon.")
    response = completion(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": f"{instruction}\n\nQuery: {query}"}],
        temperature=0.0,
        api_key=userdata.get('GEMINI_API_KEY')
    )
    return f"{query} {response.choices[0].message.content.strip()}"

def retrieve_chunks(collection, expanded_query, doc_type=None, n_results=5):
    results = collection.query(
        query_texts=[expanded_query], n_results=n_results,
        where={"doc_type": doc_type} if doc_type else None
    )
    ids = results['ids'][0]
    formatted = [
        f"--- ID: {cid} (Source: {m['source']}) ---\n{txt}"
        for cid, m, txt in zip(ids, results['metadatas'][0], results['documents'][0])
    ]
    return "\n\n".join(formatted), ids

def generate_cited_answer(query, context_str):
    response = completion(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": (
                "Synthesize an answer using ONLY the provided context. Cite the source chunk ID "
                "in [brackets] after every factual claim. If the context doesn't support an answer, "
                "say 'INFO NOT IN CONTEXT' rather than filling gaps with general knowledge."
            )},
            {"role": "user", "content": f"CONTEXT:\n{context_str}\n\nQUERY: {query}"}
        ],
        temperature=0.0,
        api_key=userdata.get('GEMINI_API_KEY')
    )
    return response.choices[0].message.content

def verify_citations(answer, context_str):
    # NEW — this was missing entirely. This is your grounding safety net.
    prompt = f"""Check whether every bracketed citation [chunk_id] in ANSWER is genuinely\nsupported by its matching source in CONTEXT. A citation naming a source that only lists a\nsection number, without stating the actual fact claimed, counts as unsupported.\n\nCONTEXT:\n{context_str}\n\nANSWER:\n{answer}\n\nRespond in strict JSON only, no markdown fences:\n{{"grounded": true/false, "unsupported_claims": ["..."]}}"""
    response = completion(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        api_key=userdata.get('GEMINI_API_KEY')
    )
    raw = re.sub(r'^```json\s*|\s*```$', '', response.choices[0].message.content.strip())
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"grounded": None, "unsupported_claims": [f"verifier returned non-JSON: {raw[:200]}"]}

def _query_cache_key(query, doc_type):
    return hashlib.md5(f"{query}::{doc_type}".encode()).hexdigest()

def load_query_cache(query_cache_path):
    if os.path.exists(query_cache_path):
        with open(query_cache_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {}

def save_query_cache(query_cache_path, cache):
    with open(query_cache_path, 'w', encoding='utf-8') as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

def ask(collection, query, query_cache_path, doc_type=None, force_refresh=False):
    """Answers a query, using a persisted cache so reruns (e.g. after hitting
    a rate limit mid-run) don't re-spend API calls on queries already answered."""
    cache = load_query_cache(query_cache_path)
    key = _query_cache_key(query, doc_type)

    if not force_refresh and key in cache:
        result = cache[key]
        print(f"\n{'='*60}\nQ: {result['query']}  [CACHED]\n{'='*60}")
        print(f"A: {result['answer']}")
        print(f"Grounded: {result['grounded']} | Sources: {result['chunks_used']}")
        return result

    expanded = expand_query(query, doc_type)
    context, ids = retrieve_chunks(collection, expanded, doc_type)

    if not len(ids):
        result = {"query": query, "answer": "No relevant information found.",
                  "grounded": True, "unsupported_claims": [], "chunks_used": []}
    else:
        answer = generate_cited_answer(query, context)
        verification = verify_citations(answer, context)
        result = {
            "query": query,
            "answer": answer,
            "grounded": verification.get("grounded"),
            "unsupported_claims": verification.get("unsupported_claims", []),
            "chunks_used": list(ids),
        }

    print(f"\n{'='*60}\nQ: {result['query']}\n{'='*60}")
    print(f"A: {result['answer']}")
    print(f"Grounded: {result['grounded']} | Sources: {result['chunks_used']}")
    if result["unsupported_claims"]:
        print(f"⚠️  Unsupported: {result['unsupported_claims']}")

    cache[key] = result
    save_query_cache(query_cache_path, cache)
    return result

# ---------------------------------------------------------
# 3. EXECUTION
# ---------------------------------------------------------
if __name__ == "__main__":
    # Step 1 — verify Drive is REALLY mounted before touching any paths
    ensure_drive_mounted()

    base_path = "/content/drive/MyDrive/ET_Industrial_data" # Updated path
    print(f"📍 Using base_path: {base_path}")
    print(f"📍 Full resolved path check: {os.path.abspath(base_path)}")

    # Step 2 — verify base_path actually has your real data before proceeding
    verify_base_path(base_path)

    data_dirs = [os.path.join(base_path, "regulatory"), os.path.join(base_path, "synthetic")]
    cache_path = os.path.join(base_path, "master_chunks.json")
    chroma_path = os.path.join(base_path, "vectorstore")
    query_cache_path = os.path.join(base_path, "query_cache.json")
    os.makedirs(chroma_path, exist_ok=True)

    print(f"🚀 Initializing Phase 2 Pipeline at: {base_path}")
    db_collection = build_vector_store(data_dirs, cache_path, chroma_path)

    test_queries = [
        ("What safety measures apply to fencing of machinery?", "regulatory"),
        ("What is the maintenance history of PUMP-14?", "maintenance_logs"),
        ("What permits were active for confined space work, and what regulation applies?", None),
        ("what's the evacuation time for a level-4 fire", None),
    ]

    results = []
    for query, doc_type in test_queries:
        try:
            results.append(ask(db_collection, query, query_cache_path, doc_type=doc_type))
        except Exception as e:
            # Rate limit or any API failure — stop cleanly, keep what's cached.
            # Rerunning this cell will skip everything already answered above.
            print(f"\n🛑 Stopped on query '{query}': {e}")
            print("   Already-answered queries are cached — rerun this cell to resume.")
            break

✅ Google Drive verified as mounted at /content/drive
📍 Using base_path: /content/drive/MyDrive/ET_Industrial_data
📍 Full resolved path check: /content/drive/MyDrive/ET_Industrial_data
✅ base_path verified — found 5 PDF(s) in /content/drive/MyDrive/ET_Industrial_data/regulatory
🚀 Initializing Phase 2 Pipeline at: /content/drive/MyDrive/ET_Industrial_data
📁 Loading existing text database from Drive cache...
✅ Skipping unchanged file: dgms_2017_loto_procedures.pdf
✅ Skipping unchanged file: dgms_2020_04_gas_hazards.pdf
✅ Skipping unchanged file: dgms_2020_05_oil_gas_safety.pdf
✅ Skipping unchanged file: factories_act_1948.pdf
✅ Skipping unchanged file: oisd-gdn-207.pdf
✅ Skipping unchanged file: industrial_records.json


/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


🚀 No changes detected. Skipping ingestion entirely.
📊 Total chunks in collection: 513

Q: What safety measures apply to fencing of machinery?  [CACHED]
A: Based on the provided context, the safety measures that apply to the fencing of machinery in a factory include the following:

### 1. Machinery Requiring Secure Fencing
The following machinery parts must be securely fenced by safeguards of substantial construction [chunk_78]:
*   Every moving part of a prime mover and every flywheel connected to a prime mover, regardless of whether they are in the engine house [chunk_78].
*   The headrace and tailrace of every water-wheel and water turbine [chunk_78].
*   Any part of a stock-bar that projects beyond the head stock of a lathe [chunk_78].
*   The following parts, unless they are positioned or constructed in a way that makes them as safe to every employee as they would be if securely fenced:
    *   Every part of an electric generator, motor, or rotary converter [chunk_78].
    *   Ever